# DSA5208 Assignment 1.1 Normal Conditions

### Group Members

- Ashley Lee Ruoxin
- Koh Si Xing
- Kevin Royce Thomson

## 1. Project Setup

In [1]:
# Only run if needed
# %pip install pymongo

In [2]:
from pymongo import MongoClient, ReadPreference
from pymongo.errors import ExecutionTimeout, PyMongoError, ServerSelectionTimeoutError
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern

import time
import pandas as pd

### 1.1 Database Connections
Two MongoDB clients are created for different purposes:

- `client` connects to the three-node replica set, `dsa5208`. It is used for normal database operations and experiments involving replica-set-level read and write concerns. With MongoDB’s default settings, writes are routed to the primary node.
- `client3` connects directly to `mongo3` using `directConnection=true`. This connection is used to inspect `mongo3`’s local data and control or pause its replication during selected experiments.

Read and write concerns are specified separately in the `CONFIGS` dictionary below. This allows the experiments to compare different consistency and availability configurations without changing the connection setup.

In [3]:
REPLICA_SET_URI = "mongodb://mongo1:27017,mongo2:27017,mongo3:27017/?replicaSet=dsa5208"
MONGO3_URI = "mongodb://mongo3:27017/?directConnection=true"
DB_NAME = "university"

client = MongoClient(REPLICA_SET_URI, serverSelectionTimeoutMS=5000)
client3 = MongoClient(MONGO3_URI, serverSelectionTimeoutMS=5000)

### 1.2 Consistency Configurations

The experiments compare four combinations of MongoDB write concern and read concern. Write concern controls when MongoDB confirms a write, while read concern controls which version of data a read may return.

| Configuration | Write Concern | Read Concern | Meaning |
|---|---|---|---|
| `majority` | `majority` | `majority` | Confirm writes only after a majority of nodes store them; return safely majority-committed data. |
| `majority-local` | `majority` | `local` | Confirm writes after a majority stores them; return the data currently available on the selected replica. |
| `w1-majority` | `1` | `majority` | Confirm writes after only the primary stores them; return safely majority-committed data. |
| `w1-local` | `1` | `local` | Confirm writes after only the primary stores them; return the selected replica’s locally available data. |

For this three-node replica set, a majority means two nodes.

The experiments use `target="secondary"` for reads. Therefore, a read can be served by either secondary replica. When `mongo3` is paused, it may contain an older value than the primary or the other secondary.

In summary:

- `w="majority"` provides stronger write durability but may take longer.
- `w=1` confirms writes faster but replicas may not have received them yet.
- `readConcern="majority"` returns a safe majority-committed value, but not necessarily the latest value written by the same client.
- `readConcern="local"` returns the selected node’s current value immediately, which may be stale.

In [4]:
CONFIGS = {
    "majority": {"write_concern": "majority", "read_concern": "majority"},
    "majority-local": {"write_concern": "majority", "read_concern": "local"},
    "w1-majority": {"write_concern": 1, "read_concern": "majority"},
    "w1-local": {"write_concern": 1, "read_concern": "local"}
}

READ_TIMEOUT_MS = 3000   # how long a read may wait before we record TIMEOUT
NUM_READS = 10           # reads per loop, so we hit both secondaries
CATCHUP_SECONDS = 2      # safety margin for mongo3 to catch up after resuming
TIMEOUT = "TIMEOUT"

### 1.3 Relationship to Client-Centric Consistency


The configurations are used to evaluate the following client-centric consistency models:

- **Read-your-writes consistency:** After a client performs a write, a later read should return the written value or a newer value. This is more likely with `majority` write concern, but may be violated when using `w1` writes and reading from a lagging secondary.

- **Monotonic-reads consistency:** Once a client has observed a particular version of the data, subsequent reads should not return an older version. This may be violated when using `local` read concern if consecutive reads are routed to replicas with different replication delays.

- **Monotonic-writes consistency:** A client’s writes should be applied in the order in which they were issued. MongoDB’s primary normally preserves this order, but replication delays, retries, or node failures may affect the observed results.

- **Writes-follow-reads consistency:** A write issued after a read should follow the version of the data observed by that read. This is best evaluated using causal sessions and by checking whether a read from one replica is followed by a write that has observed the same or a newer version.

The `majority` configuration is expected to provide the strongest consistency guarantees. The `w1-local` configuration is expected to be the most vulnerable to stale reads and consistency violations, especially during replication delays, node failures, or network partitions. These are experimental expectations rather than absolute guarantees, because the results also depend on read preference, session usage, operation ordering, and failure conditions.

### 1.4 Helper Functions

#### 1.4.1 Checking Cluster Status


The following helper functions verify the current state of the MongoDB replica set before each experiment. They identify the primary node, display the state of all replica-set members, and confirm that `mongo3` is a secondary.

This check is important because the experiments may pause or disrupt `mongo3`. The experiment should not begin if `mongo3` is currently acting as the primary.

In [5]:
def show_members():
    """Print each member's state and whether it has applied everything the primary has."""
    members = client.admin.command("replSetGetStatus")["members"]
    primary = next((m for m in members if m["stateStr"] == "PRIMARY"), None)

    print("\n")
    
    for m in members:
        print(m["name"], m["stateStr"], m.get("optimeDate"))

    print("\n")


def check_cluster():
    """Confirm the replica set is healthy and mongo3 is a secondary."""
    hello = client.admin.command("hello")
    print("Replica set:\t\t", hello.get("setName"))
    print("Primary:\t\t", hello.get("primary"))
    print("Writable primary:\t", hello.get("isWritablePrimary"))
    show_members()

    # The experiments pause mongo3, which only makes sense if it is a secondary.
    assert hello.get("primary") != "mongo3:27017", (
        "mongo3 is currently primary. Step it down before running the experiments."
    )

#### 1.4.2 Simulating Replication Delay on `mongo3`

The following helper functions temporarily pause and resume the application of replicated operations on `mongo3`.

`stall_mongo3()` enables MongoDB’s `rsSyncApplyStop` fail point. This stops `mongo3` from applying newly replicated operations while allowing the primary and other replica-set members to continue operating. The two-second delay gives the node time to fall behind.

`unstall_mongo3()` disables the fail point and allows `mongo3` to resume applying replicated operations. The node can then catch up with the primary.

This setup is used to simulate replication lag and investigate how different consistency configurations behave when a secondary is temporarily behind.

In [6]:
def stall_mongo3():
    """Pause replication on mongo3."""
    client3.admin.command({
        "configureFailPoint": "rsSyncApplyStop",
        "mode": "alwaysOn"
    })
    time.sleep(2)


def unstall_mongo3():
    """Resume replication on mongo3."""
    client3.admin.command({
        "configureFailPoint": "rsSyncApplyStop",
        "mode": "off"
    })

## 2. Experiment Support

### 2.1 Experiment Setup

#### Creating Collections for a Configuration

The following function creates collection handles using a selected consistency configuration and read target. This allows the experiments to perform operations under different combinations of write concern, read concern, and read preference.

In [7]:
def get_collection(collection_name, config_name, target="secondary"):
    """
    Return a collection handle with the config's write and read concern applied.

    target:
        "primary"   - reads come from the primary
        "secondary" - reads come from any secondary (mongo2 or mongo3)
        "mongo3"    - reads come directly from mongo3, using local read concern,
                      so we see exactly what mongo3 has stored

    Writes always go to the primary, whatever the target.
    """
    config = CONFIGS[config_name]

    if target == "mongo3":
        db = client3.get_database(DB_NAME, read_concern=ReadConcern("local"))
        return db[collection_name]

    read_preference = {
        "primary": ReadPreference.PRIMARY,
        "secondary": ReadPreference.SECONDARY,
    }[target]

    db = client.get_database(
        DB_NAME,
        write_concern=WriteConcern(w=config["write_concern"]),
        read_concern=ReadConcern(config["read_concern"]),
        read_preference=read_preference,
    )
    return db[collection_name]

#### Preparing Test Data

The helper functions below create and reset the student and course documents used in the experiments.

`SETUP_WRITE_CONCERN` is configured with `w=3`, meaning MongoDB waits for acknowledgement from all three replica-set members before confirming the reset operation. This ensures that every node, including `mongo3`, begins each experiment with the same baseline data before replication is paused.

The `reset_document()` function creates a new document or replaces an existing document with the same identifier. It uses the strong setup write concern to ensure that the reset is replicated across all nodes.

The `make_student()` and `make_course()` functions generate standard test documents with default values. Additional student fields can be changed using keyword overrides when required.

Each consistency configuration uses unique document identifiers, such as `S001-majority` or `C001-w1-local`. This prevents data and results from different configurations from being mixed.

In [8]:
SETUP_WRITE_CONCERN = WriteConcern(w=3, wtimeout=10000)


def reset_document(collection_name, id_field, document):
    """Create or overwrite a test document and wait until every node has it."""
    db = client.get_database(DB_NAME, write_concern=SETUP_WRITE_CONCERN)
    db[collection_name].replace_one({id_field: document[id_field]}, document, upsert=True)


def make_student(student_id, name, **overrides):
    student = {
        "student_id": student_id,
        "student_name": name,
        "student_status": "Undergraduate",
        "seniority": "Year 1",
        "academic_load": "Full Time",
    }
    student.update(overrides)
    return student


def make_course(course_id, name):
    return {
        "course_id": course_id,
        "course_name": name,
        "course_status": "ACTIVE",
        "vacancy": 30,
    }

### 2.2 Shared Reading and Consistency Reporting Functions

In [9]:
def read_field(collection, query, field, session=None):
    """
    Read one field of one document.
    Returns TIMEOUT if the node could not serve the read within READ_TIMEOUT_MS
    (this is how a causal session blocks a stale read).
    """
    try:
        doc = collection.find_one(query, session=session, max_time_ms=READ_TIMEOUT_MS)
    except ExecutionTimeout:
        return TIMEOUT
    except PyMongoError as error:
        return f"ERROR ({type(error).__name__})"
    return doc.get(field) if doc else "NOT FOUND"


def read_field_repeatedly(collection, query, field, session=None, n=NUM_READS):
    return [read_field(collection, query, field, session) for _ in range(n)]


def show(label, value):
    print(f"{label:<30} {value}")


def show_reads(label, reads):
    """Print the reads as a table, one row per read."""
    table = pd.DataFrame({"read": range(1, len(reads) + 1), "value": reads})
    print(f"{label}")
    print(table.to_string(index=False))


def header(title, config_name):
    config = CONFIGS[config_name]
    print(f"=== {title} | w={config['write_concern']}, "
          f"readConcern={config['read_concern']} ===")


def report_stale_reads(reads, expected):
    """Read-your-writes check: any value other than what we wrote is stale."""
    stale = sum(1 for r in reads if r not in (expected, TIMEOUT))
    timeouts = reads.count(TIMEOUT)
    verdict = "VIOLATION" if stale else "no violation in this run"
    print(f"RESULTS: {verdict}: {stale} stale, {timeouts} timed out, out of {len(reads)} reads")


def report_regressions(reads, order):
    """Monotonic-reads check: count reads that went back to an older value."""
    rank = {value: i for i, value in enumerate(order)}
    newest_seen = -1
    regressions = 0
    for value in reads:
        if value not in rank:  # TIMEOUT or error
            continue
        if rank[value] < newest_seen:
            regressions += 1
        newest_seen = max(newest_seen, rank[value])

    timeouts = reads.count(TIMEOUT)
    verdict = "VIOLATION" if regressions else "no violation in this run"
    print(f"RESULTS: {verdict}: {regressions} went backwards, {timeouts} timed out, "
          f"out of {len(reads)} reads")

### 2.3 Consistency Experiment Functions

#### Read-Your-Writes Experiment

This experiment tests whether a client can read a value immediately after writing it. The test compares repeated secondary reads performed without a session against reads performed within a causally consistent session.

1. Create or reset a configuration-specific student document with an initial `student_status` of `Undergraduate`.
2. Pause replication on `mongo3` to simulate replication delay.
3. Update the student’s status to `Graduate` without using a session.
4. Perform repeated reads from a secondary node and record the returned values.
5. Use `report_stale_reads()` to check whether any read fails to return `Graduate`.
6. Display the replica-set status to observe the state of the nodes.
7. Start a causally consistent MongoDB session.
8. Update the student’s status to `Post Doc` within the session.
9. Perform repeated secondary reads using the same session.
10. Use `report_stale_reads()` to check whether any read fails to return `Post Doc`.
11. Compare the results from the non-session and causal-session tests, including any stale reads, errors, or timeouts.
12. Resume replication on `mongo3` in the `finally` block so that it can catch up before the next experiment.

In [10]:
def run_read_your_writes(config_name):
    header("Read-your-writes", config_name)
    students_secondary = get_collection("students", config_name, "secondary")

    student_id = f"S001-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(student_id, "Robert Tan"))

    stall_mongo3()
    try:
        # Without a session
        students_secondary.update_one(query, {"$set": {"student_status": "Graduate"}})
        show("Wrote student_status:", "Graduate")
        reads = read_field_repeatedly(students_secondary, query, "student_status")
        show_reads("Reads (no session):", reads)
        report_stale_reads(reads, expected="Graduate")

        show_members()

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            students_secondary.update_one(
                query, {"$set": {"student_status": "Post Doc"}}, session=session
            )
            show("Wrote student_status:", "Post Doc")
            reads = read_field_repeatedly(
                students_secondary, query, "student_status", session=session
            )
            show_reads("Reads (causal consistency session):", reads)
            report_stale_reads(reads, expected="Post Doc")
    except:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

#### Monotonic-Reads Experiment

This experiment tests whether a client’s repeated reads return data that does not move backwards to an older version. The student’s seniority is updated through several versions, and the results are compared when reading from secondary nodes with and without a causally consistent session.

1. Create or reset a configuration-specific student document with an initial seniority of `Year 1`.
2. Pause replication on `mongo3` to simulate replication delay.
3. Issue the following seniority updates sequentially through the primary:
   - Update to `Year 2`.
   - Update to `Year 3`.
   - Update to `Year 4`.
   - Update to `Year 5`.
4. Perform repeated reads of the student’s seniority from secondary nodes without using a session.
5. Display the returned values and use `report_regressions()` to check whether a later read returns an older seniority value.
6. Repeat the reads using a causally consistent MongoDB session.
7. Display and analyse the session-based results, including any regressions or timeouts.
8. Compare the read sequences with the expected order: `Year 1` → `Year 2` → `Year 3` → `Year 4` → `Year 5`.
9. Resume replication on `mongo3` in the `finally` block so that it can catch up before the next run.


In [11]:
SENIORITY_ORDER = ["Year 1", "Year 2", "Year 3", "Year 4", "Year 5"]


def run_monotonic_reads(config_name):
    header("Monotonic reads", config_name)
    students_secondary = get_collection("students", config_name, "secondary")

    student_id = f"S002-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(student_id, "Feyd Johnston"))

    stall_mongo3()
    try:
        for year in SENIORITY_ORDER[1:]:
            students_secondary.update_one(query, {"$set": {"seniority": year}})
        show("Wrote seniority:", " -> ".join(SENIORITY_ORDER))

        # Without a session
        reads = read_field_repeatedly(students_secondary, query, "seniority")
        show_reads("Reads (no session):", reads)
        report_regressions(reads, SENIORITY_ORDER)

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            reads = read_field_repeatedly(
                students_secondary, query, "seniority", session=session
            )
        show_reads("Reads (causal consistency session):", reads)
        report_regressions(reads, SENIORITY_ORDER)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

#### Monotonic-Writes Experiment

This experiment tests whether a client’s writes are applied and observed in the same order in which they were issued. The student’s academic history is updated sequentially with two qualifications, and the results are compared across the primary, a stalled secondary, and the recovered `mongo3` node.

1. Create or reset a configuration-specific student document with an initial academic history containing only `B.Sc. Computer Science`.
2. Pause replication on `mongo3` to simulate replication delay.
3. Issue the following writes sequentially through the primary:
   - Add `M.Sc. Data Science`.
   - Add `PhD Information Systems`.
4. Read the academic history from the primary to confirm the order of the writes.
5. Read the academic history directly from the stalled `mongo3` to observe its delayed state.
6. Perform repeated reads from secondary nodes without using a session.
7. Display the results and compare the order of the qualifications observed by the replicas.
8. Resume replication on `mongo3`.
9. Wait for `mongo3` to catch up with the primary.
10. Read from `mongo3` again and verify whether the qualifications appear in the same order as they were written.

In [12]:
def run_monotonic_writes(config_name):
    header("Monotonic writes", config_name)
    students_primary = get_collection("students", config_name, "primary")
    students_secondary = get_collection("students", config_name, "secondary")
    students_mongo3 = get_collection("students", config_name, "mongo3")

    student_id = f"S003-{config_name}"
    query = {"student_id": student_id}
    reset_document("students", "student_id", make_student(
        student_id, "Alan White",
        student_status="Post Doc",
        academic_history=["B.Sc. Computer Science"],
    ))

    stall_mongo3()
    try:
        for degree in ["M.Sc. Data Science", "PhD Information Systems"]:
            students_primary.update_one(query, {"$push": {"academic_history": degree}})

        show("Primary:", read_field(students_primary, query, "academic_history"))
        show("mongo3 (stalled):", read_field(students_mongo3, query, "academic_history"))

        reads = read_field_repeatedly(students_secondary, query, "academic_history")
        show_reads("Reads from secondaries (no session):", reads)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

    time.sleep(CATCHUP_SECONDS)
    show("mongo3 (resumed):", read_field(students_mongo3, query, "academic_history"))

#### Writes-Follow-Reads Experiment

This experiment tests whether a write issued after a read correctly follows the version of the data observed by that read. The experiment reads a course’s status and then writes a student’s enrollment note based on the observed status.

1. Create or reset a configuration-specific course document with an initial status of `ACTIVE`.
2. Create or reset a configuration-specific student document with an empty `enrollment_note`.
3. Pause replication on `mongo3` to simulate replication delay.
4. Without using a session, read the course status from a secondary node.
5. Write a student enrollment note through the primary using the course status returned by the read.
6. Read the course status and enrollment note from `mongo3` and the primary to compare their visible states.
7. Start a causally consistent MongoDB session.
8. Read the course status from a secondary using the session.
9. Write the student enrollment note through the primary using the value returned by the session read.
10. Read the enrollment note again from a secondary using the same session.
11. Compare the non-session and session-based results to determine whether the write correctly follows the preceding read.
12. Resume replication on `mongo3` in the `finally` block so that it can catch up before the next experiment.



In [13]:
def run_writes_follow_reads(config_name):
    header("Writes-follow-reads", config_name)
    courses_secondary = get_collection("courses", config_name, "secondary")
    courses_mongo3 = get_collection("courses", config_name, "mongo3")
    students_primary = get_collection("students", config_name, "primary")
    students_secondary = get_collection("students", config_name, "secondary")
    students_mongo3 = get_collection("students", config_name, "mongo3")

    course_id = f"C001-{config_name}"
    course_name = "Programming Methodology"
    student_id = f"S004-{config_name}"
    course_query = {"course_id": course_id}
    student_query = {"student_id": student_id}

    reset_document("courses", "course_id", make_course(course_id, course_name))
    reset_document("students", "student_id",
                   make_student(student_id, "Feyd Johnston", enrollment_note=None))

    stall_mongo3()
    try:
        # Without a session: read the course, then write a note based on that read
        status = read_field(courses_secondary, course_query, "course_status")
        show("Read course_status:", status)
        students_primary.update_one(student_query, {"$set": {
            "enrollment_note": f"Enrolled in {course_name} while course_status was: {status}"
        }})

        show("mongo3 course_status:", read_field(courses_mongo3, course_query, "course_status"))
        show("mongo3 enrollment_note:", read_field(students_mongo3, student_query, "enrollment_note"))
        show("Primary enrollment_note:", read_field(students_primary, student_query, "enrollment_note"))

        # With a causally consistent session
        with client.start_session(causal_consistency=True) as session:
            status = read_field(courses_secondary, course_query, "course_status", session=session)
            show("Session read course_status:", status)
            students_primary.update_one(student_query, {"$set": {
                "enrollment_note": f"Session enrollment, course_status: {status}"
            }}, session=session)
            note = read_field(students_secondary, student_query, "enrollment_note", session=session)
            show("Session read-back:", note)
    except ServerSelectionTimeoutError as error:
        print("Experiment stopped")
    finally:
        unstall_mongo3()

### 2.4 Checking the Cluster

Before starting a new experiment, replication on `mongo3` is resumed in case a previous experiment was interrupted while the node was stalled.

The cluster status is then checked to confirm that:

- the replica set is available;
- a primary node is elected;
- `mongo3` is operating as a secondary;
- the replica-set members are reporting their current states.

```python
unstall_mongo3()
check_cluster()

In [14]:
unstall_mongo3()
check_cluster()

Replica set:		 dsa5208
Primary:		 mongo2:27017
Writable primary:	 True


mongo1:27017 SECONDARY 2026-09-19 04:06:11
mongo2:27017 PRIMARY 2026-09-19 04:06:11
mongo3:27017 SECONDARY 2026-09-19 04:06:11




## Experiment 1: Read-Your-Writes Consistency

After the client updates the student’s `student_status` through the primary, repeated reads are performed from secondary nodes. The returned values are checked to determine whether the client can observe its own write or whether a secondary still returns the previous value due to replication delay.


### Predictions

**Read-your-writes** requires that, after a client writes a value, its later reads return that value or a newer value.

**Note:** Session refers to a causally consistent session.

| Configuration | Without a Session | With a Causally Consistent Session |
|---|---|---|
| `majority` | **May be violated.** The write is durable after reaching a majority, but reads are directed to secondaries. A lagging secondary can return the earlier majority-committed value instead of the client’s latest write. | **Expected to be upheld.** The session associates later reads with the previous write. A lagging secondary should wait or return `TIMEOUT` instead of returning an older value. |
| `majority-local` | **May be violated.** The write reaches a majority, but local reads return the selected secondary’s current value immediately. A paused or lagging secondary can therefore return the old value. | **Expected to be upheld.** The session should prevent completed reads from returning a value older than the preceding write, although lagging replicas may cause timeouts. |
| `w1-majority` | **May be violated.** The write is acknowledged after the primary stores it, before it necessarily reaches any secondary. A majority read can return the earlier majority-committed value while the new write is still replicating. | **Expected to be upheld.** The session should prevent completed reads from returning an older value, but reads may wait or time out while replicas catch up. |
| `w1-local` | **Most likely to be violated.** The write is acknowledged by the primary immediately, and local reads return whichever value is stored on the selected secondary. This creates the largest opportunity for a secondary to return stale data. | **Expected to be upheld.** The session should prevent completed reads from returning a value older than the preceding write; a lagging secondary may return `TIMEOUT` instead. |

### Majority Configuration






In [15]:
run_read_your_writes("majority")

=== Read-your-writes | w=majority, readConcern=majority ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1      Graduate
    2 Undergraduate
    3      Graduate
    4 Undergraduate
    5      Graduate
    6 Undergraduate
    7      Graduate
    8 Undergraduate
    9 Undergraduate
   10 Undergraduate
RESULTS: VIOLATION: 6 stale, 0 timed out, out of 10 reads


mongo1:27017 SECONDARY 2026-09-19 04:06:15
mongo2:27017 PRIMARY 2026-09-19 04:06:17
mongo3:27017 SECONDARY 2026-09-19 04:06:15


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1  TIMEOUT
    2  TIMEOUT
    3 Post Doc
    4  TIMEOUT
    5  TIMEOUT
    6 Post Doc
    7 Post Doc
    8 Post Doc
    9 Post Doc
   10  TIMEOUT
RESULTS: no violation in this run: 0 stale, 5 timed out, out of 10 reads


#### Interpretation
- **Secondary reads without a session**

  - **Result**: A few secondary reads returned `Undergraduate` after the update to `Graduate` was acknowledged. This is a read-your-writes violation.

  - **Why it happened**: `mongo3` was paused and still held `Undergraduate`. Although `Graduate` had reached a majority of nodes, the old value was also safely majority committed during the reset operation. A normal majority read can therefore return this older value.

- **Secondary reads with a causally consistent session**

  - **Result**: No stale values were returned. Completed reads returned `Post Doc`, while some reads returned `TIMEOUT`.

  - **Why it happened**: The causal session required each read to be at least as recent as the preceding `Post Doc` write. A lagging secondary could not return an older value, so MongoDB waited and returned `TIMEOUT` when the node could not catch up.

### Majority-Local Configuration


In [16]:
run_read_your_writes("majority-local")

=== Read-your-writes | w=majority, readConcern=local ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1 Undergraduate
    2 Undergraduate
    3 Undergraduate
    4 Undergraduate
    5      Graduate
    6 Undergraduate
    7      Graduate
    8 Undergraduate
    9 Undergraduate
   10 Undergraduate
RESULTS: VIOLATION: 8 stale, 0 timed out, out of 10 reads


mongo1:27017 SECONDARY 2026-09-19 04:06:35
mongo2:27017 PRIMARY 2026-09-19 04:06:35
mongo3:27017 SECONDARY 2026-09-19 04:06:32


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1  TIMEOUT
    2 Post Doc
    3  TIMEOUT
    4 Post Doc
    5 Post Doc
    6 Post Doc
    7  TIMEOUT
    8 Post Doc
    9  TIMEOUT
   10 Post Doc
RESULTS: no violation in this run: 0 stale, 4 timed out, out of 10 reads


#### Interpretation

- **Secondary reads without a session**

  - **Result:** Stale reads were observed after the update to `Graduate` was acknowledged, indicating a read-your-writes violation.

  - **Why it happened:** The `majority-local` configuration uses a majority write concern but `readConcern="local"`. The update had reached the primary and another secondary, while paused `mongo3` still held `Undergraduate`. Local reads return the value currently stored on the selected secondary immediately, so reads routed to the lagging node could return the old value.

- **Secondary reads with a causally consistent session**

  - **Result:** The completed session-based reads returned `Post Doc`, while some reads timed out. No stale values were observed in this run.

  - **Why it happened:** The causal session carries the ordering of the preceding `Post Doc` write. When a selected secondary could not serve a sufficiently recent read, the operation waited and eventually returned `TIMEOUT` rather than returning an older value. The exact number of successful reads and timeouts may differ between runs, depending on which secondary serves each read.

### w1-Majority Configuration

In [17]:
run_read_your_writes("w1-majority")

=== Read-your-writes | w=1, readConcern=majority ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1 Undergraduate
    2 Undergraduate
    3 Undergraduate
    4 Undergraduate
    5 Undergraduate
    6 Undergraduate
    7 Undergraduate
    8 Undergraduate
    9 Undergraduate
   10 Undergraduate
RESULTS: VIOLATION: 10 stale, 0 timed out, out of 10 reads


mongo1:27017 SECONDARY 2026-09-19 04:06:47
mongo2:27017 PRIMARY 2026-09-19 04:06:49
mongo3:27017 SECONDARY 2026-09-19 04:06:47


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1 Post Doc
    2  TIMEOUT
    3 Post Doc
    4  TIMEOUT
    5  TIMEOUT
    6  TIMEOUT
    7 Post Doc
    8 Post Doc
    9 Post Doc
   10 Post Doc
RESULTS: no violation in this run: 0 stale, 4 timed out, out of 10 reads


#### Interpretation

- **Secondary reads without a session**

  - **Result:** All secondary reads returned the earlier value, `Undergraduate`, after the update to `Graduate` was acknowledged. This is a read-your-writes violation.

  - **Why it happened:** The `w1-majority` configuration uses `w=1`, so MongoDB acknowledged the update once the primary stored it. The update had not necessarily reached a majority of nodes when the secondary reads began. Since `readConcern="majority"` returns majority-committed data, the secondaries returned the earlier value, `Undergraduate`, which had already been replicated during the reset operation. Paused `mongo3` further ensured that at least one secondary remained behind.

- **Secondary reads with a causally consistent session**

  - **Result:** No stale values were observed among completed reads, but most reads timed out.

  - **Why it happened:** The session tracked the preceding `Post Doc` write and prevented a secondary from returning an older value. Because the selected secondary could not provide a sufficiently recent version while `mongo3` was stalled, the read waited and returned `TIMEOUT`. This demonstrates the trade-off between avoiding stale reads and maintaining read availability.

### w1-Local Configuration

In [18]:
run_read_your_writes("w1-local")

=== Read-your-writes | w=1, readConcern=local ===
Wrote student_status:          Graduate
Reads (no session):
 read         value
    1 Undergraduate
    2      Graduate
    3 Undergraduate
    4      Graduate
    5 Undergraduate
    6 Undergraduate
    7      Graduate
    8 Undergraduate
    9      Graduate
   10      Graduate
RESULTS: VIOLATION: 5 stale, 0 timed out, out of 10 reads


mongo1:27017 SECONDARY 2026-09-19 04:07:01
mongo2:27017 PRIMARY 2026-09-19 04:07:03
mongo3:27017 SECONDARY 2026-09-19 04:07:01


Wrote student_status:          Post Doc
Reads (causal consistency session):
 read    value
    1  TIMEOUT
    2 Post Doc
    3  TIMEOUT
    4  TIMEOUT
    5 Post Doc
    6  TIMEOUT
    7 Post Doc
    8  TIMEOUT
    9  TIMEOUT
   10 Post Doc
RESULTS: no violation in this run: 0 stale, 6 timed out, out of 10 reads


#### Interpretation

- **Secondary reads without a session**

  - **Result:** Stale reads were observed after the update to `Graduate` was acknowledged, indicating a read-your-writes violation.

  - **Why it happened:** The `w1-local` configuration acknowledges the write as soon as the primary stores it, while local reads return the value currently available on the selected secondary. Because `mongo3` was paused and other replicas may still have been catching up, secondary reads could return the earlier value, `Undergraduate`.

- **Secondary reads with a causally consistent session**

  - **Result:** No stale values were observed among completed reads, but some reads timed out.

  - **Why it happened:** The session associated each read with the preceding `Post Doc` write. When a selected secondary could not provide a sufficiently recent value, the read waited and returned `TIMEOUT` rather than returning an older value. This run therefore avoided stale reads at the cost of reduced read availability.

## Experiment 2: Monotonic-Reads Consistency

The student’s `seniority` is updated sequentially from `Year 1` to `Year 5` while replication on `mongo3` is paused. The document is then read repeatedly from secondary nodes, both without a session and with a causally consistent session.

**What to look for:** A monotonic-read violation occurs when a later read returns an older seniority value than a value already observed, for example:

```text
Year 4 → Year 5 → Year 2

### Predictions

**Monotonic reads** requires that once a client has observed a value, later completed reads must not return an older value.

**Note:** Session refers to a causally consistent session.

| Configuration | Without a Session | With a Causally Consistent Session |
|---|---|---|
| `majority` | **May be violated.** Reads can be routed to different secondaries. An up-to-date secondary may return a newer value, while stalled `mongo3` can return an older majority-committed value. | **Expected to be upheld.** Once the session has observed a newer value, a lagging secondary should wait or return `TIMEOUT` rather than return an older value. |
| `majority-local` | **May be violated.** Local reads return the current value stored on the selected secondary. Switching between an up-to-date secondary and stalled `mongo3` can cause later reads to return an older value. | **Expected to be upheld.** The session should prevent completed reads from moving backwards; a lagging secondary may return `TIMEOUT`. |
| `w1-majority` | **May be violated or remain stale.** The primary acknowledges writes before they reach a majority. Secondary reads may repeatedly return the same earlier majority-committed value, which is stale but not itself a regression. A violation occurs only if the client first sees a newer value and later sees an older one. | **Expected to be upheld.** The session should avoid completed reads that go backwards, although delayed replicas may cause timeouts. |
| `w1-local` | **Most likely to be violated.** Writes are acknowledged by the primary immediately, while local secondary reads can return whichever version is currently stored on the selected node. This makes switching between newer and older replicas especially likely. | **Expected to be upheld.** The session should prevent completed reads from returning an older value after a newer one has been observed; lagging replicas may return `TIMEOUT`. |

### Majority Configuration

In [19]:
run_monotonic_reads("majority")

=== Monotonic reads | w=majority, readConcern=majority ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 1
    2 Year 1
    3 Year 5
    4 Year 1
    5 Year 1
    6 Year 5
    7 Year 5
    8 Year 1
    9 Year 5
   10 Year 1
RESULTS: VIOLATION: 4 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read   value
    1  Year 5
    2  Year 5
    3  Year 5
    4  Year 5
    5 TIMEOUT
    6 TIMEOUT
    7 TIMEOUT
    8  Year 5
    9 TIMEOUT
   10 TIMEOUT
RESULTS: no violation in this run: 0 went backwards, 5 timed out, out of 10 reads


#### Interpretation

- **Secondary reads without a session**

  - **Result:** Read regressions were observed, indicating a monotonic-reads violation. After returning `Year 5`, later reads returned the older value, `Year 1`.

  - **Why it happened:** `mongo3` was paused after the updates, so it remained at `Year 1`, while the primary and an available secondary reached `Year 5`. Because reads were directed to any secondary, the driver could alternate between an up-to-date secondary and stalled `mongo3`. Although `readConcern="majority"` returns safe data, it does not ensure that each later read is at least as recent as a previous client read.

- **Secondary reads with a causally consistent session**

  - **Result:** No read regressions were observed among completed reads. The session initially returned `Year 1`, then returned `Year 5` for later completed reads; some operations returned `TIMEOUT`.

  - **Why it happened:** The causal session tracked the newest value already observed by the client. After the session observed `Year 5`, a lagging secondary could not return `Year 1`, so MongoDB waited and returned `TIMEOUT` when the node could not catch up. This preserved monotonic reads, although some read availability was lost while `mongo3` was stalled.

### Majority-Local Configuration

In [20]:
run_monotonic_reads("majority-local")

=== Monotonic reads | w=majority, readConcern=local ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 5
    2 Year 1
    3 Year 5
    4 Year 5
    5 Year 5
    6 Year 1
    7 Year 1
    8 Year 1
    9 Year 5
   10 Year 5
RESULTS: VIOLATION: 4 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read   value
    1  Year 1
    2  Year 1
    3  Year 5
    4  Year 5
    5 TIMEOUT
    6  Year 5
    7  Year 5
    8 TIMEOUT
    9  Year 5
   10  Year 5
RESULTS: no violation in this run: 0 went backwards, 2 timed out, out of 10 reads



#### Interpretation

- **Secondary reads without a session**

  - **Result:** Read regressions were observed. The sequence alternated between `Year 5` and the older `Year 1`, which is a monotonic-reads violation.

  - **Why it happened:** The writes reached a majority of nodes, but `readConcern="local"` returned whatever value was immediately available on each selected secondary. The up-to-date secondary returned `Year 5`, while paused `mongo3` returned its older local value, `Year 1`.

- **Secondary reads with a causally consistent session**

  - **Result:** No read regressions were observed among completed reads, although some reads returned `TIMEOUT`.

  - **Why it happened:** After the session observed `Year 5`, a lagging secondary could not provide an older version for later reads. Instead of returning `Year 1`, the operation waited and timed out when the selected node could not catch up.

### w1-Majority Configuration

In [21]:
run_monotonic_reads("w1-majority")

=== Monotonic reads | w=1, readConcern=majority ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 1
    2 Year 1
    3 Year 1
    4 Year 1
    5 Year 1
    6 Year 1
    7 Year 1
    8 Year 1
    9 Year 1
   10 Year 1
RESULTS: no violation in this run: 0 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read   value
    1  Year 5
    2 TIMEOUT
    3 TIMEOUT
    4 TIMEOUT
    5  Year 5
    6  Year 5
    7 TIMEOUT
    8  Year 5
    9  Year 5
   10  Year 5
RESULTS: no violation in this run: 0 went backwards, 4 timed out, out of 10 reads


#### Interpretation

- **Secondary reads without a session**

  - **Result:** No monotonic-read violation was observed. All 10 reads returned `Year 1`, the value present before any of the writes were applied.

  - **Why it happened:** With `w=1`, each write is acknowledged as soon as the primary stores it, without waiting for replication to the secondaries. `readConcern="majority"` only returns a value once it can be confirmed as majority-committed, and with `mongo3` stalled for the whole run, only the primary and `mongo2` were available to form that majority. The short delay before `mongo2`'s progress was actually confirmed as majority-committed meant that, at the time these reads ran, no node had anything newer than `Year 1` that it was allowed to return. The data was stale throughout, but it never moved backwards, since it never moved at all, there was no progress for a later read to regress from.

- **Secondary reads with a causally consistent session**

  - **Result:** No read regressions were observed among completed reads, although several reads returned `TIMEOUT`.

  - **Why it happened:** The session's timestamp required any later read to be at least as recent as what the session had already observed. Once a read returned `Year 5`, a subsequent read routed to a secondary that could not yet meet that timestamp, such as the stalled `mongo3`, blocked and returned `TIMEOUT` rather than an older value.

### w1-Local Configuration

In [23]:
run_monotonic_reads("w1-local")

=== Monotonic reads | w=1, readConcern=local ===
Wrote seniority:               Year 1 -> Year 2 -> Year 3 -> Year 4 -> Year 5
Reads (no session):
 read  value
    1 Year 1
    2 Year 1
    3 Year 5
    4 Year 1
    5 Year 1
    6 Year 1
    7 Year 5
    8 Year 5
    9 Year 1
   10 Year 5
RESULTS: VIOLATION: 4 went backwards, 0 timed out, out of 10 reads
Reads (causal consistency session):
 read   value
    1  Year 5
    2 TIMEOUT
    3  Year 5
    4  Year 5
    5  Year 5
    6 TIMEOUT
    7  Year 5
    8 TIMEOUT
    9 TIMEOUT
   10  Year 5
RESULTS: no violation in this run: 0 went backwards, 4 timed out, out of 10 reads


#### Interpretation

- **Secondary reads without a session**

  - **Result:** 4 of 10 reads went backwards, a clear monotonic-reads violation.

  - **Why it happened:** `w=1` acknowledges each write as soon as the primary stores it, without waiting for replication, and `readConcern="local"` returns whatever value is currently stored on the node a read happens to land on, with no freshness check at all. With `mongo3` stalled, reads that landed on it returned `Year 1`, while reads that landed on `mongo2` returned `Year 5`. Since `ReadPreference.SECONDARY` picks a node at random for each read, the sequence the client observed jumped back and forth between the two values depending purely on which secondary happened to answer.

- **Secondary reads with a causally consistent session**

  - **Result:** No read regressions were observed among completed reads, though 4 of 10 reads returned `TIMEOUT`.

  - **Why it happened:** Once a read returned `Year 5`, the session's timestamp required any later read to be at least that recent. `mongo3` could not satisfy this while stalled, so a read routed to it blocked and returned `TIMEOUT` instead of the older value.

## Experiment 3: Monotonic-Writes Consistency

Two qualifications are appended sequentially to the student's `academic_history` while replication on `mongo3` is paused. The academic history is then compared across the primary, the stalled replica, and `mongo3` after replication resumes.

**What to look for:** Whether `mongo3` applies the qualifications in the same order as the primary after it catches up. An incomplete history while `mongo3` is paused indicates replication delay; a reversed qualification order would indicate a monotonic-writes violation.

### Predictions

The monotonic-writes experiment checks whether the two qualifications are applied to `academic_history` in the same order in which the client issued them.

| Configuration | Prediction with a Causally Consistent Session |
|---|---|
| `majority` | **Expected to be upheld.** A session does not change the order in which the primary records or replicas apply the sequential writes. |
| `majority-local` | **Expected to be upheld.** The session does not affect the replication order of the writes. |
| `w1-majority` | **Expected to be upheld.** The session does not affect the primary’s write order or the replication-log order. |
| `w1-local` | **Expected to be upheld.** The session does not alter the order of the sequential writes. |



### Majority Configuration

In [27]:
run_monotonic_writes("majority")

=== Monotonic writes | w=majority, readConcern=majority ===
Primary:                       ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                                                                 value
    1 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    2 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    3                                              [B.Sc. Computer Science]
    4                                              [B.Sc. Computer Science]
    5 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    6                                              [B.Sc. Computer Science]
    7                                              [B.Sc. Computer Science]
    8 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    9 [B.Sc. Computer Science, M.Sc. Data Science, PhD

#### Interpretation

- **Primary and stalled `mongo3`**

  - **Result:** The primary contained the complete academic history in the expected order, while stalled `mongo3` contained only the initial qualification.

  - **Why it happened:** Replication on `mongo3` was paused before the new qualifications were applied. The primary recorded both `$push` operations, but `mongo3` could not apply them while stalled. This is replication delay, not a monotonic-writes violation.

- **Secondary reads during replication delay**

  - **Result:** Secondary reads returned either the complete academic history or the earlier incomplete history.

  - **Why it happened:** `ReadPreference.SECONDARY` allowed each read to be served by either an up-to-date secondary or stalled `mongo3`. The up-to-date secondary returned the complete history, while paused `mongo3` returned only the initial qualification because it had not yet applied either new write.

- **`mongo3` after replication resumed**

  - **Result:** After replication resumed, `mongo3` contained the complete academic history in the same order as the primary. No monotonic-writes violation was observed.

  - **Why it happened:** The primary recorded the two `$push` operations in sequence in its replication log. When `mongo3` resumed, it applied the queued operations in that same order: first the M.Sc. qualification, then the PhD qualification. Replication delay postponed the writes but did not reorder them.

### Majority-Local Configuration

In [28]:
run_monotonic_writes("majority-local")

=== Monotonic writes | w=majority, readConcern=local ===
Primary:                       ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                                                                 value
    1                                              [B.Sc. Computer Science]
    2                                              [B.Sc. Computer Science]
    3                                              [B.Sc. Computer Science]
    4 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    5 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    6                                              [B.Sc. Computer Science]
    7                                              [B.Sc. Computer Science]
    8                                              [B.Sc. Computer Science]
    9                                              [B.Sc.

#### Interpretation

- **Primary and stalled `mongo3`**

  - **Result:** The primary contained the complete academic history in the expected order, while stalled `mongo3` contained only the initial qualification.

  - **Why it happened:** The `majority-local` configuration waited for the writes to reach a majority of nodes, but `mongo3` was paused and therefore could not apply the new qualifications. The primary and an available secondary were sufficient to acknowledge the writes.

- **Secondary reads during replication delay**

  - **Result:** Secondary reads returned either the complete academic history or the earlier incomplete history.

  - **Why it happened:** `readConcern="local"` immediately returned the value stored on the selected secondary. Reads served by an up-to-date secondary returned the complete history, while reads served by stalled `mongo3` returned only the initial qualification. This demonstrates replication delay, not a monotonic-writes violation.

- **`mongo3` after replication resumed**

  - **Result:** After replication resumed, `mongo3` contained the complete academic history in the same order as the primary. No monotonic-writes violation was observed.

  - **Why it happened:** MongoDB retained the sequential `$push` operations in the primary’s replication log. When `mongo3` resumed, it applied the delayed operations in their original order, preserving the M.Sc.-then-PhD sequence.

### w1-Majority Configuration

In [29]:
run_monotonic_writes("w1-majority")

=== Monotonic writes | w=1, readConcern=majority ===
Primary:                       ['B.Sc. Computer Science']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                    value
    1 [B.Sc. Computer Science]
    2 [B.Sc. Computer Science]
    3 [B.Sc. Computer Science]
    4 [B.Sc. Computer Science]
    5 [B.Sc. Computer Science]
    6 [B.Sc. Computer Science]
    7 [B.Sc. Computer Science]
    8 [B.Sc. Computer Science]
    9 [B.Sc. Computer Science]
   10 [B.Sc. Computer Science]
mongo3 (resumed):              ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']


#### Interpretation

- **Primary and stalled `mongo3`**

  - **Result:** Both the primary and stalled `mongo3` initially returned only the initial qualification.

  - **Why it happened:** The `w1-majority` configuration acknowledges each `$push` operation after the primary stores it, without waiting for replication. However, the reads use `readConcern="majority"`, which returns only data that has reached a majority of nodes. The new qualifications had not yet become majority committed, so the primary read returned the earlier baseline history.

- **Secondary reads during replication delay**

  - **Result:** All secondary reads returned only the initial academic history.

  - **Why it happened:** The secondary nodes had not yet applied the new qualifications, and the majority read concern did not expose updates that had not reached a majority. This shows delayed visibility of `w=1` writes, not a monotonic-writes violation.

- **`mongo3` after replication resumed**

  - **Result:** After replication resumed, `mongo3` contained the complete academic history in the correct order. No monotonic-writes violation was observed.

  - **Why it happened:** Once replication resumed, the delayed `$push` operations reached `mongo3` and were applied in the order issued by the primary: first the M.Sc. qualification, then the PhD qualification. The writes were initially invisible to majority reads, but they were not reordered.

### w1-Local Configuration

In [ ]:
run_monotonic_writes("w1-local")

=== Monotonic writes | w=1, readConcern=local ===
Primary:                       ['B.Sc. Computer Science', 'M.Sc. Data Science', 'PhD Information Systems']
mongo3 (stalled):              ['B.Sc. Computer Science']
Reads from secondaries (no session):
 read                                                                 value
    1 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    2 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    3 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    4                                              [B.Sc. Computer Science]
    5                                              [B.Sc. Computer Science]
    6 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    7 [B.Sc. Computer Science, M.Sc. Data Science, PhD Information Systems]
    8                                              [B.Sc. Computer Science]
    9 [B.Sc. Computer Science, M.Sc. Data Science, PhD Informati

#### Interpretation

- **Primary and stalled `mongo3`**

  - **Result:** The primary returned the complete academic history, while stalled `mongo3` returned only the initial qualification.

  - **Why it happened:** The `w1-local` configuration acknowledged the writes as soon as the primary stored them. Since `readConcern="local"` returns the value currently stored on the selected node, the primary immediately showed both new qualifications while paused `mongo3` remained at the initial history.

- **Secondary reads during replication delay**

  - **Result:** Secondary reads returned either the complete academic history or the earlier incomplete history.

  - **Why it happened:** Reads were served by different secondary nodes. An up-to-date secondary returned the complete history, while stalled `mongo3` returned only the initial qualification. This shows that `w=1` writes can become visible on replicas at different times.

- **`mongo3` after replication resumed**

  - **Result:** After replication resumed, `mongo3` contained the complete academic history in the same order as the primary. No monotonic-writes violation was observed.

  - **Why it happened:** `mongo3` applied the delayed `$push` operations in the order recorded by the primary: first the M.Sc. qualification, then the PhD qualification. The delay affected visibility but did not reorder the writes.

## Experiment 4: Writes-Follow-Reads Consistency

The client reads a course’s `course_status` and then writes a student `enrollment_note` containing the value returned by that read. The experiment compares this read-then-write sequence with and without a causally consistent session.

**What to look for:** Whether the enrollment note correctly reflects the course status observed by the client, and whether a causally consistent session prevents later reads from returning data older than the preceding read or write. A lagging replica may return `TIMEOUT` rather than satisfy the causal dependency.

### Predictions

The experiment checks whether the enrollment note correctly follows the course status that the client previously read.

| Configuration | Without a Session | With a Causally Consistent Session |
|---|---|---|
| `majority` | The write is durable after reaching a majority, but a secondary read may still return an older safe course state. | The session should preserve the read-then-write order; a lagging replica may cause `TIMEOUT` rather than return an older state. |
| `majority-local` | The write reaches a majority, but local secondary reads may return a stale course state. | The session should avoid returning a state older than the session history, although reads may time out on a lagging node. |
| `w1-majority` | The enrollment note can be acknowledged before reaching a majority; majority reads may show an earlier committed state or wait. | The session should preserve the observed order where possible, but availability may decrease while a secondary is behind. |
| `w1-local` | This is most likely to expose stale data because writes are acknowledged by the primary only and reads return each selected replica's local state. | The session should prevent completed reads from returning an older value than the session requires; a lagging replica may return `TIMEOUT`. |

**Important limitation:** In the current code, the course document is reset with `w=3` before `mongo3` is paused and its `course_status` is not updated afterwards. Therefore, every node already has the course state used to create the enrollment note. The experiment can demonstrate the causal read-then-write sequence, but it is unlikely to show a node containing the enrollment note without the course state it depends on.

### Majority Configuration

In [30]:
run_writes_follow_reads("majority")

=== Writes-follow-reads | w=majority, readConcern=majority ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        Enrolled in Programming Methodology while course_status was: ACTIVE
Primary enrollment_note:       Enrolled in Programming Methodology while course_status was: ACTIVE
Session read course_status:    ACTIVE
Session read-back:             Session enrollment, course_status: ACTIVE


#### Interpretation

- **Read and write without a session**

  - **Result:** The course read returned `ACTIVE`, and the enrollment note stored on both the primary and `mongo3` correctly recorded that the course status was `ACTIVE`. No writes-follow-reads violation was observed.

  - **Why it happened:** The course document was reset with `w=3` before `mongo3` was paused, so every replica already contained `course_status = ACTIVE`. The enrollment note was then written after reading this value, and the majority write concern ensured that the note reached a majority of nodes before acknowledgement.

- **Read and write with a causally consistent session**

  - **Result:** The session read returned `ACTIVE`, and the session read-back returned an enrollment note based on `ACTIVE`. The read-then-write relationship was preserved.

  - **Why it happened:** The causally consistent session recorded the order of operations: read the course status, write the dependent enrollment note, then read the note back. The completed operations therefore respected this causal sequence.

### Majority-Local Configuration

In [31]:
run_writes_follow_reads("majority-local")

=== Writes-follow-reads | w=majority, readConcern=local ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        Enrolled in Programming Methodology while course_status was: ACTIVE
Primary enrollment_note:       Enrolled in Programming Methodology while course_status was: ACTIVE
Session read course_status:    ACTIVE
Session read-back:             Session enrollment, course_status: ACTIVE


#### Interpretation

- **Read and write without a session**

  - **Result:** The course read returned `ACTIVE`, and the enrollment note on both the primary and `mongo3` correctly recorded that the course status was `ACTIVE`. No writes-follow-reads violation was observed.

  - **Why it happened:** Although `readConcern="local"` returns the value currently stored on the selected node, all nodes already contained `course_status = ACTIVE` because the course document was reset with `w=3` before `mongo3` was paused. The majority write concern then ensured that the enrollment note reached a majority of nodes before acknowledgement.

- **Read and write with a causally consistent session**

  - **Result:** The session read returned `ACTIVE`, and the session read-back returned an enrollment note based on `ACTIVE`. The read-then-write relationship was preserved.

  - **Why it happened:** The causal session maintained the order between the course read, the dependent enrollment-note write, and the final read-back. No stale or inconsistent dependent value was observed in this run.

### w1-Majority Configuration

In [32]:
run_writes_follow_reads("w1-majority")

=== Writes-follow-reads | w=1, readConcern=majority ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        Enrolled in Programming Methodology while course_status was: ACTIVE
Primary enrollment_note:       Enrolled in Programming Methodology while course_status was: ACTIVE
Session read course_status:    ACTIVE
Session read-back:             Session enrollment, course_status: ACTIVE


### w1-Local Configuration

#### Interpretation

- **Read and write without a session**

  - **Result:** The course read returned `ACTIVE`, and the enrollment note on both the primary and `mongo3` correctly recorded that the course status was `ACTIVE`. No writes-follow-reads violation was observed.

  - **Why it happened:** With `w=1`, the enrollment note was acknowledged after the primary stored it, without waiting for replication to a majority. However, in this run `mongo3` also received the note, and all nodes already contained `course_status = ACTIVE` from the earlier `w=3` reset. Therefore, the dependent note and course state remained consistent.

- **Read and write with a causally consistent session**

  - **Result:** The session read returned `ACTIVE`, and the session read-back returned an enrollment note based on `ACTIVE`. The read-then-write relationship was preserved.

  - **Why it happened:** The session preserved the causal order between reading the course status, writing the dependent note, and reading the note back. No stale or inconsistent dependent value was observed in this run.

In [33]:
run_writes_follow_reads("w1-local")

=== Writes-follow-reads | w=1, readConcern=local ===
Read course_status:            ACTIVE
mongo3 course_status:          ACTIVE
mongo3 enrollment_note:        None
Primary enrollment_note:       Enrolled in Programming Methodology while course_status was: ACTIVE
Session read course_status:    ACTIVE
Session read-back:             Session enrollment, course_status: ACTIVE


#### Interpretation

- **Read and write without a session**

  - **Result:** `mongo3` returned `ACTIVE` for the course status but `None` for the enrollment note, while the primary contained the new note.

  - **Why it happened:** The course document was replicated to all nodes during the initial `w=3` reset. After `mongo3` was paused, the enrollment note was written with `w=1`, so MongoDB acknowledged it once the primary stored it. The stalled `mongo3` had not yet received the later note and therefore returned `None`.

- **Read and write with a causally consistent session**

  - **Result:** The session read returned `ACTIVE`, and the session read-back returned an enrollment note based on `ACTIVE`. No writes-follow-reads violation was observed.

  - **Why it happened:** The session preserved the causal order between reading the course status, writing the dependent enrollment note, and reading the note back. The note correctly reflected the course state observed by the client.